# Retriever Evaluation

In [3]:
import sys
sys.path.insert(0, '/Users/skyler/Projects/document_retrieval_project')

from loader import load_data
from retrievers.bm25 import BM25Retriever
from retrievers.tf_idf_retriever import TFIDFRetriever
from retrievers.dense_retriever import DenseRetriever
from retrievers.hybrid_retriever import HybridRetriever
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time
import pandas as pd

In [4]:
# Build corpus
ds = load_data()
passages_text = []
queries = []
for example in ds:
    queries.append(example["query"])
    for passage in example["passages"]["passage_text"]:
        passages_text.append(passage)

print(f"Total passages: {len(passages_text)}")
print(f"Total queries: {len(queries)}")

TypeError: string indices must be integers, not 'str'

In [ ]:
# Fit retrievers
bm25 = BM25Retriever(top_k=10)
tfidf = TFIDFRetriever(top_k=10)
dense = DenseRetriever(top_k=10)
hybrid = HybridRetriever(top_k=10)

bm25.fit(passages_text)
tfidf.fit(passages_text)
dense.fit("sbert_embeddings.npy")
hybrid.fit(passages_text)

print("All retrievers fitted")

In [ ]:
# Run MRR@10 evaluation
MAX_QUERIES = 1_000_000

retrievers = {"BM25": bm25, "TF-IDF": tfidf, "Dense": dense, "Hybrid": hybrid}
results = []

for name, retriever in retrievers.items():
    ds = load_data()
    mrr = mrr_at_10(retriever, ds, max_queries=MAX_QUERIES)
    avg_time = measure_retrieval_time(retriever, queries[:MAX_QUERIES])
    results.append({"Retriever": name, "MRR@10": round(mrr, 4), "Avg ms/query": round(avg_time, 3)})
    print(f"{name} done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time:.3f}")

In [ ]:
# Display and save results
df = pd.DataFrame(results)
df.to_csv("evaluations.csv", index=False)
print(df.to_string(index=False))

## Test Retrievers Individually

In [ ]:
from retrievers.hybrid_retriever import HybridRetriever

# Run MRR@10 evaluation
MAX_QUERIES = 1_000_000
ds = load_data()
hybrid = HybridRetriever(top_k=10, candidate_k=100)
name = "Hybrid"

passages_text = []
queries = []
for example in ds:
    queries.append(example["query"])
    for passage in example["passages"]["passage_text"]:
        passages_text.append(passage)

hybrid.fit(passages_text)
mrr = mrr_at_10(hybrid, ds, max_queries=MAX_QUERIES)
avg_time = measure_retrieval_time(hybrid, queries[:MAX_QUERIES])
print(f"{name} done — MRR@10: {mrr:.4f}, Avg ms/query: {avg_time:.3f}")